In [ ]:
!apt-get update

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:14 http://

In [ ]:
!apt-get install -y openmpi-bin libopenmpi-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
libopenmpi-dev set to manually installed.
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 132 not upgraded.


In [1]:
%%writefile lab4.mpi.c
#include <mpi.h>
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

#define TAG01 100
#define TAG10 101
#define TAGR 200
#define TAG_RANGE 1000

void run_pingpong(int rank);
void run_ring(int rank, int size);

void run_pingpong(int rank) {
    int msg_sizes[] = {1, 8, 64, 512, 4096};
    int num_sizes = 5;
    int K = 10000;

    if (rank == 0) {
        printf("Message Size (bytes) | Iterations | Avg RTT (micro seconds) | Est. One-way (micro seconds)\n");
        printf("--------------------------------------------------------------------\n");
    }

    for (int i = 0; i < num_sizes; i++) {
        int msg_size = msg_sizes[i];

        char *buffer = (char *)malloc(msg_size);
        if (buffer == NULL) {
            printf("Rank %d: Memory allocation failed\n", rank);
            MPI_Abort(MPI_COMM_WORLD, 1);
        }

        MPI_Status status;
        double t0, t1;

        if (rank == 0) {
            t0 = MPI_Wtime();

            for (int k = 0; k < K; k++) {
                int tag = k % TAG_RANGE;

                MPI_Send(buffer, msg_size, MPI_BYTE, 1, tag, MPI_COMM_WORLD);
                MPI_Recv(buffer, msg_size, MPI_BYTE, 1, MPI_ANY_TAG, MPI_COMM_WORLD, &status);

                if (status.MPI_TAG != tag) {
                    printf("Rank 0: Tag mismatch at iteration %d (expected %d, got %d)\n",
                           k, tag, status.MPI_TAG);
                }
            }

            t1 = MPI_Wtime();

            double total_time = t1 - t0;
            double avg_rtt_us = (total_time / K) * 1000000.0;
            double one_way_us = avg_rtt_us / 2.0;

            printf("%20d | %10d | %12.3f | %17.3f\n",
                   msg_size, K, avg_rtt_us, one_way_us);
        }

        else if (rank == 1) {
            for (int k = 0; k < K; k++) {
                int tag = k % TAG_RANGE;

                MPI_Recv(buffer, msg_size, MPI_BYTE, 0, MPI_ANY_TAG, MPI_COMM_WORLD, &status);

                if (status.MPI_TAG != tag) {
                    printf("Rank 1: Tag mismatch at iteration %d (expected %d, got %d)\n",
                           k, tag, status.MPI_TAG);
                }

                MPI_Send(buffer, msg_size, MPI_BYTE, 0, tag, MPI_COMM_WORLD);
            }
        }

        free(buffer);
    }
}

void run_ring(int rank, int size) {
    int right = (rank + 1) % size;
    int left  = (rank - 1 + size) % size;

    int send_value = rank;
    int recv_value = -1;

    MPI_Status status;

    MPI_Sendrecv(&send_value, 1, MPI_INT, right, TAGR,
                 &recv_value, 1, MPI_INT, left,  TAGR,
                 MPI_COMM_WORLD, &status);

    printf("Rank %d: sent %d to rank %d, received %d from rank %d\n",
           rank, send_value, right, recv_value, left);
}

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (argc < 2) {
        if (rank == 0)
            printf("Usage: %s pingpong|ring\n", argv[0]);
        MPI_Finalize();
        return 0;
    }

    if (strcmp(argv[1], "pingpong") == 0) {
        if (size != 2) {
            if (rank == 0) printf("Ping-pong requires -np 2\n");
        } else {
            run_pingpong(rank);
        }
    }
    else if (strcmp(argv[1], "ring") == 0) {
        if (size < 2) {
            if (rank == 0)
                printf("Ring requires -np >= 2\n");
        }
        else {
            run_ring(rank, size);
        }
    }

    MPI_Finalize();
    return 0;
}

Writing lab4.mpi.c


In [2]:
!mpicc -O3 lab4.mpi.c -o lab4

In [3]:
!mpirun --allow-run-as-root --oversubscribe -np 2 ./lab4 pingpong

Message Size (bytes) | Iterations | Avg RTT (micro seconds) | Est. One-way (micro seconds)
--------------------------------------------------------------------
                   1 |      10000 |        1.093 |             0.547
                   8 |      10000 |        1.073 |             0.536
                  64 |      10000 |        1.053 |             0.526
                 512 |      10000 |        1.505 |             0.753
                4096 |      10000 |        5.803 |             2.902


In [4]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./lab4 ring

Rank 0: sent 0 to rank 1, received 3 from rank 3
Rank 1: sent 1 to rank 2, received 0 from rank 0
Rank 2: sent 2 to rank 3, received 1 from rank 1
Rank 3: sent 3 to rank 0, received 2 from rank 2
